In [4]:
# ==========================================================
# Cell 1 — Upload DNS ZIP, extract, pick largest CSV, show head/tail (FAST)
# ==========================================================
import os, zipfile, glob
import pandas as pd
from google.colab import files

# ---- 1) Upload ZIP
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
ZIP_PATH = zip_name
print(f"✅ Uploaded ZIP: {ZIP_PATH} | size: {os.path.getsize(ZIP_PATH)} bytes")

# ---- 2) Extract (safe-ish) with basename fallback (like your conn notebook)
EXTRACT_DIR = "extracted_dns_csvs"
os.makedirs(EXTRACT_DIR, exist_ok=True)

def safe_extract(zip_path, extract_dir):
    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.infolist():
            member_path = os.path.normpath(os.path.join(extract_dir, member.filename))
            if not os.path.abspath(member_path).startswith(os.path.abspath(extract_dir)):
                raise RuntimeError(f"Unsafe path in zip: {member.filename}")
        z.extractall(extract_dir)

try:
    safe_extract(ZIP_PATH, EXTRACT_DIR)
    print("✅ Extracted to:", EXTRACT_DIR)
except RuntimeError as e:
    print("⚠️", e)
    print("🔧 Retrying with basename-only extraction...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        for member in z.infolist():
            if member.is_dir():
                continue
            out_path = os.path.join(EXTRACT_DIR, os.path.basename(member.filename))
            with z.open(member, "r") as src, open(out_path, "wb") as dst:
                dst.write(src.read())
    print("✅ Extracted (basename-only) to:", EXTRACT_DIR)

# ---- 3) Pick largest CSV
csvs = glob.glob(os.path.join(EXTRACT_DIR, "*.csv"))
if not csvs:
    raise SystemExit("⛔ No CSV files found inside the ZIP.")

sizes = sorted([(c, os.path.getsize(c)) for c in csvs], key=lambda x: x[1], reverse=True)
DATA_CSV = sizes[0][0]
print(f"\n📌 Found CSVs: {len(csvs)}")
print("📌 Selected largest CSV:", DATA_CSV, "|", f"{sizes[0][1]/1024/1024:.2f} MB")

# ---- 4) HEAD(5)
df_head = pd.read_csv(DATA_CSV, nrows=5)
print("\n🔎 Preview: HEAD(5)")
display(df_head)

# ---- 5) TAIL(5) — FAST: read only last N rows by chunking
N_TAIL = 5
CHUNK = 200_000  # adjust if needed

tail_chunks = []
for chunk in pd.read_csv(DATA_CSV, chunksize=CHUNK):
    tail_chunks.append(chunk)
    if len(tail_chunks) > 3:   # keep only last ~3 chunks in memory
        tail_chunks.pop(0)

df_tail = pd.concat(tail_chunks, ignore_index=True).tail(N_TAIL)

print("\n🔎 Preview: TAIL(5)")
display(df_tail)

print("\n✅ DATA_CSV set to:", DATA_CSV)
print("✅ Columns:", list(df_head.columns))


Saving merged (3).zip to merged (3) (2).zip
✅ Uploaded ZIP: merged (3) (2).zip | size: 14805138 bytes
✅ Extracted to: extracted_dns_csvs

📌 Found CSVs: 1
📌 Selected largest CSV: extracted_dns_csvs/merged (3).csv | 352.16 MB

🔎 Preview: HEAD(5)


,@timestamp,source.ip,zeek.dns.query,zeek.dns.rcode_name,zeek.dns.qtype_name,dns.question.subdomain,zeek.dns.rtt,zeek.dns.TTLs,zeek.dns.answers,network.transport,zeek.dns.rejected,zeek.dns.TC,destination.ip,destination.port
0,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,AAAA,-,-,-,-,udp,False,False,192.168.100.5,53
1,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,A,-,0.001,"3,600",192.168.100.112,udp,False,False,192.168.100.5,53
2,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,AAAA,-,-,-,-,udp,False,False,192.168.100.5,53
3,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,A,-,0.001,"3,600",192.168.100.112,udp,False,False,192.168.100.5,53
4,"Sep 22, 2025 @ 15:29:59.203",192.168.100.20,xmsecu.com,-,A,-,-,-,-,udp,False,False,217.218.127.127,53



🔎 Preview: TAIL(5)


,@timestamp,source.ip,zeek.dns.query,zeek.dns.rcode_name,zeek.dns.qtype_name,dns.question.subdomain,zeek.dns.rtt,zeek.dns.TTLs,zeek.dns.answers,network.transport,zeek.dns.rejected,zeek.dns.TC,destination.ip,destination.port
436315,"Oct 20, 2025 @ 12:00:00.875",192.168.100.5,asia.pool.ntp.org,REFUSED,A,asia.pool,-,-,-,udp,False,False,77.104.104.104,53
436316,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,NOERROR,AAAA,asia.pool,-,-,-,udp,False,False,8.8.8.8,53
436317,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,NOERROR,AAAA,asia.pool,-,-,-,udp,False,False,8.8.8.8,53
436318,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,-,A,asia.pool,-,-,-,udp,False,False,8.8.8.8,53
436319,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,-,A,asia.pool,-,-,-,udp,False,False,8.8.8.8,53



✅ DATA_CSV set to: extracted_dns_csvs/merged (3).csv
✅ Columns: ['@timestamp', 'source.ip', 'zeek.dns.query', 'zeek.dns.rcode_name', 'zeek.dns.qtype_name', 'dns.question.subdomain', 'zeek.dns.rtt', 'zeek.dns.TTLs', 'zeek.dns.answers', 'network.transport', 'zeek.dns.rejected', 'zeek.dns.TC', 'destination.ip', 'destination.port']


In [7]:
# ==========================================================
# Cell 2 — Load & Clean DNS raw logs (Zeek DNS) [FIX v2]
# - Robust timestamp parsing for "Sep 22, 2025 @ 15:29:59.592"
# - Normalize columns
# - IPv4-only + remove 0.0.0.0
# - Build df_clean (no aggregation yet)
# ==========================================================
import pandas as pd
import numpy as np
import re
import ipaddress
from math import log2

print("🔧 Loading DNS CSV (low_memory=False)...")
df = pd.read_csv(DATA_CSV, low_memory=False)

# --- Normalize column names (strip spaces + remove BOM)
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace("\ufeff", "", regex=False)
)
print("🧾 Normalized columns:", list(df.columns))
print("✅ Loaded:", df.shape)

# --- Required columns check
required_cols = [
    "@timestamp","source.ip","destination.ip","destination.port",
    "zeek.dns.query","zeek.dns.rcode_name","zeek.dns.qtype_name",
    "dns.question.subdomain","zeek.dns.rtt","zeek.dns.TTLs","zeek.dns.answers",
    "network.transport","zeek.dns.rejected","zeek.dns.TC"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise SystemExit(f"⛔ Missing required columns: {missing}")

# --- Strip key fields
for c in ["@timestamp","source.ip","destination.ip"]:
    df[c] = df[c].astype(str).str.strip()

# ==========================================================
# 1) Timestamp parsing (FIXED)
# ==========================================================
# Format example: "Sep 22, 2025 @ 15:29:59.592"
# We'll remove the " @ " to make it: "Sep 22, 2025 15:29:59.592"
ts_raw = df["@timestamp"].astype(str)

# normalize spacing around @
ts_norm = ts_raw.str.replace(r"\s*@\s*", " ", regex=True)

# parse with explicit format (fast + stable)
# %b %d, %Y %H:%M:%S.%f  (Sep 22, 2025 15:29:59.592)
df["ts"] = pd.to_datetime(ts_norm, format="%b %d, %Y %H:%M:%S.%f", errors="coerce")

# if there are rows without milliseconds, fallback parse them too
bad = df["ts"].isna()
if bad.any():
    df.loc[bad, "ts"] = pd.to_datetime(
        ts_norm[bad],
        format="%b %d, %Y %H:%M:%S",
        errors="coerce"
    )

before = len(df)
df = df.dropna(subset=["ts"]).copy()
print(f"🕒 Timestamp parsed. Dropped {before - len(df)} rows with invalid ts.")
print("🕒 Time range:", df["ts"].min(), "→", df["ts"].max())

# window_start (1-minute)
df["window_start"] = df["ts"].dt.floor("min")

# ==========================================================
# 2) IP filtering: remove IPv6 + 0.0.0.0
# ==========================================================
def is_ipv4(s):
    try:
        return ipaddress.ip_address(str(s)).version == 4
    except:
        return False

before = len(df)
df = df[df["source.ip"].apply(is_ipv4)].copy()
df = df[df["destination.ip"].apply(is_ipv4)].copy()
df = df[(df["source.ip"] != "0.0.0.0") & (df["destination.ip"] != "0.0.0.0")].copy()
print(f"🧹 IPv4-only + remove 0.0.0.0: {before} → {len(df)} rows")

# ==========================================================
# 3) Normalize simple fields
# ==========================================================
df["qtype"] = df["zeek.dns.qtype_name"].astype(str).str.strip().str.upper().replace({"-": "UNKNOWN", "NAN": "UNKNOWN"})
df["rcode"] = df["zeek.dns.rcode_name"].astype(str).str.strip().str.upper().replace({"-": "UNKNOWN", "NAN": "UNKNOWN"})
df["transport"] = df["network.transport"].astype(str).str.strip().str.lower().replace({"-": "unknown", "nan": "unknown"})

df["destination.port"] = pd.to_numeric(df["destination.port"], errors="coerce").fillna(0).astype(int)

def to_bool(x):
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    return s in ["true", "t", "1", "yes", "y"]

df["is_rejected"] = df["zeek.dns.rejected"].apply(to_bool).astype(int)
df["is_tc"] = df["zeek.dns.TC"].apply(to_bool).astype(int)

# ==========================================================
# 4) Parse RTT + TTLs + answers count
# ==========================================================
df["rtt_s"] = pd.to_numeric(
    df["zeek.dns.rtt"].astype(str).str.strip().str.replace(",", ""),
    errors="coerce"
).fillna(0.0)

def parse_ttls(x):
    s = str(x).strip()
    if s.lower() in ["-", "nan", "none", ""]:
        return []
    nums = re.findall(r"\d+", s.replace(",", ""))
    return [int(n) for n in nums] if nums else []

ttls = df["zeek.dns.TTLs"].apply(parse_ttls)
df["ttl_count"] = ttls.apply(len)
df["ttl_avg"] = ttls.apply(lambda a: float(np.mean(a)) if len(a) else 0.0)
df["ttl_min"] = ttls.apply(lambda a: float(np.min(a)) if len(a) else 0.0)

def parse_answers_count(x):
    s = str(x).strip()
    if s.lower() in ["-", "nan", "none", ""]:
        return 0
    if "," in s:
        return len([t for t in s.split(",") if t.strip()])
    return 1

df["answers_count"] = df["zeek.dns.answers"].apply(parse_answers_count).astype(int)

# ==========================================================
# 5) Query/Subdomain text features (for tunneling + beaconing)
# ==========================================================
df["query"] = df["zeek.dns.query"].astype(str).str.strip().str.lower().replace({"-": ""})
df["subdomain"] = df["dns.question.subdomain"].astype(str).str.strip().str.lower().replace({"-": ""})

df["query_len"] = df["query"].str.len().astype(int)
df["subdomain_len"] = df["subdomain"].str.len().astype(int)

def labels_count(q):
    q = str(q).strip().strip(".")
    if not q:
        return 0
    return len([p for p in q.split(".") if p])

df["labels_count"] = df["query"].apply(labels_count).astype(int)

def base_domain(q):
    q = str(q).strip().strip(".")
    parts = [p for p in q.split(".") if p]
    if len(parts) >= 2:
        return parts[-2] + "." + parts[-1]
    return q

df["base_domain"] = df["query"].apply(base_domain)

def shannon_entropy(s):
    s = str(s)
    if not s:
        return 0.0
    freqs = {}
    for ch in s:
        freqs[ch] = freqs.get(ch, 0) + 1
    n = len(s)
    ent = 0.0
    for c in freqs.values():
        p = c / n
        ent -= p * log2(p)
    return float(ent)

df["subdomain_entropy"] = df["subdomain"].apply(shannon_entropy)

def long_label_ratio(q):
    q = str(q).strip().strip(".")
    labels = [p for p in q.split(".") if p]
    if not labels:
        return 0.0
    long_cnt = sum(1 for l in labels if len(l) >= 30)
    return long_cnt / len(labels)

df["long_label_ratio"] = df["query"].apply(long_label_ratio)

df["is_nxdomain"] = (df["rcode"] == "NXDOMAIN").astype(int)
df["is_txt"] = (df["qtype"] == "TXT").astype(int)
df["is_any"] = (df["qtype"] == "ANY").astype(int)
df["is_null"] = (df["qtype"] == "NULL").astype(int)

# ==========================================================
# 6) Final df_clean
# ==========================================================
keep_cols = [
    "ts","window_start",
    "source.ip","destination.ip","destination.port",
    "query","subdomain","base_domain",
    "qtype","rcode","transport",
    "rtt_s","ttl_count","ttl_avg","ttl_min","answers_count",
    "query_len","subdomain_len","labels_count","subdomain_entropy","long_label_ratio",
    "is_nxdomain","is_rejected","is_tc","is_txt","is_any","is_null"
]
df_clean = df[keep_cols].copy()

print("✅ df_clean ready:", df_clean.shape)
print("🕒 df_clean time range:", df_clean["ts"].min(), "→", df_clean["ts"].max())

display(df_clean.head(5))
display(df_clean.tail(5))


🔧 Loading DNS CSV (low_memory=False)...
🧾 Normalized columns: ['@timestamp', 'source.ip', 'zeek.dns.query', 'zeek.dns.rcode_name', 'zeek.dns.qtype_name', 'dns.question.subdomain', 'zeek.dns.rtt', 'zeek.dns.TTLs', 'zeek.dns.answers', 'network.transport', 'zeek.dns.rejected', 'zeek.dns.TC', 'destination.ip', 'destination.port']
✅ Loaded: (2236320, 14)
🕒 Timestamp parsed. Dropped 0 rows with invalid ts.
🕒 Time range: 2025-09-17 10:57:10.506000 → 2025-10-20 15:59:59.087000
🧹 IPv4-only + remove 0.0.0.0: 2236320 → 1994686 rows
✅ df_clean ready: (1994686, 27)
🕒 df_clean time range: 2025-09-17 10:57:10.506000 → 2025-10-20 15:59:59.087000


,ts,window_start,source.ip,destination.ip,destination.port,query,subdomain,base_domain,qtype,rcode,...,subdomain_len,labels_count,subdomain_entropy,long_label_ratio,is_nxdomain,is_rejected,is_tc,is_txt,is_any,is_null
0,2025-09-22 15:29:59.592,2025-09-22 15:29:00,192.168.100.119,192.168.100.5,53,es.itp.local,,itp.local,AAAA,NOERROR,...,0,3,0.0,0.0,0,0,0,0,0,0
1,2025-09-22 15:29:59.592,2025-09-22 15:29:00,192.168.100.119,192.168.100.5,53,es.itp.local,,itp.local,A,NOERROR,...,0,3,0.0,0.0,0,0,0,0,0,0
2,2025-09-22 15:29:59.592,2025-09-22 15:29:00,192.168.100.119,192.168.100.5,53,es.itp.local,,itp.local,AAAA,NOERROR,...,0,3,0.0,0.0,0,0,0,0,0,0
3,2025-09-22 15:29:59.592,2025-09-22 15:29:00,192.168.100.119,192.168.100.5,53,es.itp.local,,itp.local,A,NOERROR,...,0,3,0.0,0.0,0,0,0,0,0,0
4,2025-09-22 15:29:59.203,2025-09-22 15:29:00,192.168.100.20,217.218.127.127,53,xmsecu.com,,xmsecu.com,A,UNKNOWN,...,0,2,0.0,0.0,0,0,0,0,0,0


,ts,window_start,source.ip,destination.ip,destination.port,query,subdomain,base_domain,qtype,rcode,...,subdomain_len,labels_count,subdomain_entropy,long_label_ratio,is_nxdomain,is_rejected,is_tc,is_txt,is_any,is_null
2236315,2025-10-20 12:00:00.875,2025-10-20 12:00:00,192.168.100.5,77.104.104.104,53,asia.pool.ntp.org,asia.pool,ntp.org,A,REFUSED,...,9,4,2.725481,0.0,0,0,0,0,0,0
2236316,2025-10-20 12:00:00.874,2025-10-20 12:00:00,192.168.100.9,8.8.8.8,53,asia.pool.ntp.org,asia.pool,ntp.org,AAAA,NOERROR,...,9,4,2.725481,0.0,0,0,0,0,0,0
2236317,2025-10-20 12:00:00.874,2025-10-20 12:00:00,192.168.100.9,8.8.8.8,53,asia.pool.ntp.org,asia.pool,ntp.org,AAAA,NOERROR,...,9,4,2.725481,0.0,0,0,0,0,0,0
2236318,2025-10-20 12:00:00.874,2025-10-20 12:00:00,192.168.100.9,8.8.8.8,53,asia.pool.ntp.org,asia.pool,ntp.org,A,UNKNOWN,...,9,4,2.725481,0.0,0,0,0,0,0,0
2236319,2025-10-20 12:00:00.874,2025-10-20 12:00:00,192.168.100.9,8.8.8.8,53,asia.pool.ntp.org,asia.pool,ntp.org,A,UNKNOWN,...,9,4,2.725481,0.0,0,0,0,0,0,0


In [8]:
# ==========================================================
# Cell 3 — DNS Aggregation (1m) + Feature Engineering (NO ML, NO severity)
# خروجی: agg_features (per source.ip × window_start)
# ==========================================================
import numpy as np
import pandas as pd

print("🧱 Building 1-minute aggregated DNS dataset (per source.ip × window_start)...")

# ---------- 1) Host×Minute aggregation ----------
g = df_clean.groupby(["source.ip", "window_start"], sort=False)

agg_host = g.agg(
    query_count=("query", "size"),
    uniq_queries=("query", "nunique"),
    uniq_subdomains=("subdomain", "nunique"),
    uniq_base_domains=("base_domain", "nunique"),
    uniq_resolvers=("destination.ip", "nunique"),

    nxdomain_rate=("is_nxdomain", "mean"),
    rejected_rate=("is_rejected", "mean"),
    tc_rate=("is_tc", "mean"),

    txt_rate=("is_txt", "mean"),
    any_rate=("is_any", "mean"),
    null_rate=("is_null", "mean"),

    query_len_mean=("query_len", "mean"),
    query_len_max=("query_len", "max"),

    labels_mean=("labels_count", "mean"),
    labels_max=("labels_count", "max"),

    sub_len_mean=("subdomain_len", "mean"),
    sub_len_max=("subdomain_len", "max"),

    sub_entropy_mean=("subdomain_entropy", "mean"),
    sub_entropy_max=("subdomain_entropy", "max"),

    long_label_ratio_mean=("long_label_ratio", "mean"),

    rtt_mean=("rtt_s", "mean"),
    rtt_std=("rtt_s", "std"),

    ttl_avg_mean=("ttl_avg", "mean"),
    ttl_min_mean=("ttl_min", "mean"),

    answers_cnt_mean=("answers_count", "mean"),
    answers_cnt_max=("answers_count", "max"),
).reset_index()

# std can be NaN when only one row
agg_host["rtt_std"] = agg_host["rtt_std"].fillna(0.0)

# ---------- 2) Rolling / temporal features (per host) ----------
# proxy "volume" for DNS tunneling: sum of query length per minute
# (we'll compute it from raw to be accurate)
len_sum = df_clean.groupby(["source.ip", "window_start"], sort=False)["query_len"].sum().rename("sum_query_len").reset_index()
agg_host = agg_host.merge(len_sum, on=["source.ip", "window_start"], how="left")
agg_host["sum_query_len"] = agg_host["sum_query_len"].fillna(0.0)

# sort for rolling features
agg_host = agg_host.sort_values(["source.ip", "window_start"]).reset_index(drop=True)

def add_temporal_features(x: pd.DataFrame) -> pd.DataFrame:
    # 60-minute rolling sums/means (on 1m windows)
    x["sum_query_len_roll_60m"] = x["sum_query_len"].rolling(60, min_periods=10).sum()
    x["uniq_subdomains_roll_60m"] = x["uniq_subdomains"].rolling(60, min_periods=10).sum()
    x["nxdomain_roll_60m"] = x["nxdomain_rate"].rolling(60, min_periods=10).mean()
    x["entropy_roll_60m"] = x["sub_entropy_mean"].rolling(60, min_periods=10).mean()

    # 30-minute trend (simple slope proxy): current - mean(last 30m)
    m30 = x["sum_query_len"].rolling(30, min_periods=10).mean()
    x["sum_query_len_trend_30m"] = x["sum_query_len"] - m30

    # fill early NaNs with 0
    for c in ["sum_query_len_roll_60m","uniq_subdomains_roll_60m","nxdomain_roll_60m","entropy_roll_60m","sum_query_len_trend_30m"]:
        x[c] = x[c].fillna(0.0)
    return x

# avoid pandas DeprecationWarning: operate on group values only
agg_host = agg_host.groupby("source.ip", group_keys=False).apply(add_temporal_features)

print("✅ agg_host:", agg_host.shape)
print("🕒 agg_host time range:", agg_host["window_start"].min(), "→", agg_host["window_start"].max())
display(agg_host.head(5))


# ---------- 3) Pair-based features for DNS Beaconing (source.ip × base_domain) ----------
print("🧭 Building pair-based DNS beaconing features (source.ip × base_domain)...")

pair_dom = df_clean.groupby(["source.ip", "base_domain", "window_start"], sort=False).size().rename("pair_qcount").reset_index()

# sort for beacon features
pair_dom = pair_dom.sort_values(["source.ip", "base_domain", "window_start"]).reset_index(drop=True)

def add_beacon_pair(x: pd.DataFrame) -> pd.DataFrame:
    # persistence over last 5 minutes: how many of last 5 windows had activity (>=1)
    x["active"] = (x["pair_qcount"] > 0).astype(int)
    x["persist_5m"] = x["active"].rolling(5, min_periods=1).sum()

    # regularity: based on inter-arrival times between active windows (in minutes)
    # for each row, look back last ~10 active timestamps
    t = x["window_start"].astype("datetime64[ns]").values
    active_idx = np.where(x["active"].values == 1)[0]

    reg = np.zeros(len(x), dtype=float)
    for i in range(len(x)):
        # indices of active up to i
        ai = active_idx[active_idx <= i]
        if len(ai) < 4:
            reg[i] = 0.0
            continue
        last = ai[-10:]
        tt = t[last].astype("datetime64[m]").astype(np.int64)
        diffs = np.diff(tt)
        if len(diffs) < 2:
            reg[i] = 0.0
            continue
        mu = np.mean(diffs)
        sd = np.std(diffs)
        # higher regularity when std is small vs mean
        reg[i] = float(1.0 / (1.0 + (sd / (mu + 1e-9))))
    x["regularity"] = reg

    # beacon score pair: weighted persistence + regularity (0..~1)
    x["beacon_score_pair"] = (x["persist_5m"] / 5.0) * (0.5 + 0.5 * x["regularity"])
    return x.drop(columns=["active"])

pair_dom = pair_dom.groupby(["source.ip", "base_domain"], group_keys=False).apply(add_beacon_pair)

# reduce to host×minute: pick best (max) domain each minute
idx = pair_dom.groupby(["source.ip", "window_start"])["beacon_score_pair"].idxmax()
best = pair_dom.loc[idx, ["source.ip","window_start","base_domain","beacon_score_pair","persist_5m","regularity","pair_qcount"]].copy()
best = best.rename(columns={
    "base_domain": "dns_beacon_best_domain",
    "beacon_score_pair": "dns_beacon_score",
    "persist_5m": "dns_beacon_best_persist_5m",
    "regularity": "dns_beacon_best_regularity",
    "pair_qcount": "dns_beacon_best_pair_qcount",
})

print("✅ agg_beacon_host:", best.shape)
display(best.head(5))


# ---------- 4) Merge host features + beacon features ----------
agg_features = agg_host.merge(best, on=["source.ip","window_start"], how="left")

# fill missing beacon fields (no activity / no best domain)
agg_features["dns_beacon_score"] = agg_features["dns_beacon_score"].fillna(0.0)
agg_features["dns_beacon_best_domain"] = agg_features["dns_beacon_best_domain"].fillna("")
agg_features["dns_beacon_best_pair_qcount"] = agg_features["dns_beacon_best_pair_qcount"].fillna(0).astype(int)
agg_features["dns_beacon_best_persist_5m"] = agg_features["dns_beacon_best_persist_5m"].fillna(0.0)
agg_features["dns_beacon_best_regularity"] = agg_features["dns_beacon_best_regularity"].fillna(0.0)

print("✅ agg_features (feature-only):", agg_features.shape)
display(agg_features.head(5))
display(agg_features.tail(5))


🧱 Building 1-minute aggregated DNS dataset (per source.ip × window_start)...
✅ agg_host: (60578, 34)
🕒 agg_host time range: 2025-09-17 10:57:00 → 2025-10-20 15:59:00


/tmp/ipython-input-3325586047.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_host = agg_host.groupby("source.ip", group_keys=False).apply(add_temporal_features)


,source.ip,window_start,query_count,uniq_queries,uniq_subdomains,uniq_base_domains,uniq_resolvers,nxdomain_rate,rejected_rate,tc_rate,...,ttl_avg_mean,ttl_min_mean,answers_cnt_mean,answers_cnt_max,sum_query_len,sum_query_len_roll_60m,uniq_subdomains_roll_60m,nxdomain_roll_60m,entropy_roll_60m,sum_query_len_trend_30m
0,10.10.10.18,2025-09-17 13:26:00,130,28,25,16,3,0.0,0.0,0.0,...,349.469231,127.569231,3.592308,20,2613,0.0,0.0,0.0,0.0,0.0
1,10.10.10.18,2025-09-17 13:28:00,45,4,1,4,1,0.0,0.0,0.0,...,1640.833333,1093.333333,6.666667,10,1605,0.0,0.0,0.0,0.0,0.0
2,10.10.10.18,2025-09-17 13:29:00,10,1,1,1,1,0.0,0.0,0.0,...,1762.500000,120.000000,10.000000,10,620,0.0,0.0,0.0,0.0,0.0
3,10.10.10.18,2025-09-17 13:30:00,15,1,1,1,1,0.0,0.0,0.0,...,0.000000,0.000000,3.000000,3,330,0.0,0.0,0.0,0.0,0.0
4,10.10.10.18,2025-09-17 13:31:00,30,2,2,2,1,0.0,0.0,0.0,...,1762.500000,120.000000,10.000000,10,1290,0.0,0.0,0.0,0.0,0.0


🧭 Building pair-based DNS beaconing features (source.ip × base_domain)...
✅ agg_beacon_host: (60578, 7)


/tmp/ipython-input-3325586047.py:130: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pair_dom = pair_dom.groupby(["source.ip", "base_domain"], group_keys=False).apply(add_beacon_pair)


,source.ip,window_start,dns_beacon_best_domain,dns_beacon_score,dns_beacon_best_persist_5m,dns_beacon_best_regularity,dns_beacon_best_pair_qcount
296,10.10.10.18,2025-09-17 13:26:00,_udp.local,0.100000,1.0,0.000000,10
297,10.10.10.18,2025-09-17 13:28:00,_udp.local,0.200000,2.0,0.000000,20
298,10.10.10.18,2025-09-17 13:29:00,_udp.local,0.300000,3.0,0.000000,10
299,10.10.10.18,2025-09-17 13:30:00,_udp.local,0.695518,4.0,0.738796,15
300,10.10.10.18,2025-09-17 13:31:00,_udp.local,0.871358,5.0,0.742716,15


✅ agg_features (feature-only): (60578, 39)


,source.ip,window_start,query_count,uniq_queries,uniq_subdomains,uniq_base_domains,uniq_resolvers,nxdomain_rate,rejected_rate,tc_rate,...,sum_query_len_roll_60m,uniq_subdomains_roll_60m,nxdomain_roll_60m,entropy_roll_60m,sum_query_len_trend_30m,dns_beacon_best_domain,dns_beacon_score,dns_beacon_best_persist_5m,dns_beacon_best_regularity,dns_beacon_best_pair_qcount
0,10.10.10.18,2025-09-17 13:26:00,130,28,25,16,3,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,_udp.local,0.100000,1.0,0.000000,10
1,10.10.10.18,2025-09-17 13:28:00,45,4,1,4,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,_udp.local,0.200000,2.0,0.000000,20
2,10.10.10.18,2025-09-17 13:29:00,10,1,1,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,_udp.local,0.300000,3.0,0.000000,10
3,10.10.10.18,2025-09-17 13:30:00,15,1,1,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,_udp.local,0.695518,4.0,0.738796,15
4,10.10.10.18,2025-09-17 13:31:00,30,2,2,2,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,_udp.local,0.871358,5.0,0.742716,15


,source.ip,window_start,query_count,uniq_queries,uniq_subdomains,uniq_base_domains,uniq_resolvers,nxdomain_rate,rejected_rate,tc_rate,...,sum_query_len_roll_60m,uniq_subdomains_roll_60m,nxdomain_roll_60m,entropy_roll_60m,sum_query_len_trend_30m,dns_beacon_best_domain,dns_beacon_score,dns_beacon_best_persist_5m,dns_beacon_best_regularity,dns_beacon_best_pair_qcount
60573,192.168.30.15,2025-10-20 14:59:00,20,5,5,4,1,0.2,0.0,0.0,...,83276.0,322.0,0.119780,2.102474,-356.666667,windowsupdate.com,0.654760,5.0,0.309520,4
60574,192.168.30.15,2025-10-20 15:00:00,4,1,1,1,1,0.0,0.0,0.0,...,83424.0,323.0,0.117715,2.124528,-726.133333,microsoft.com,0.869398,5.0,0.738796,4
60575,45.33.56.234,2025-10-07 15:04:00,2,1,1,1,1,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,,0.100000,1.0,0.000000,2
60576,45.79.218.137,2025-10-05 11:38:00,2,1,1,1,1,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,,0.100000,1.0,0.000000,2
60577,45.79.68.93,2025-09-27 12:41:00,3,1,1,1,1,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,,0.100000,1.0,0.000000,3


In [10]:
# ==========================================================
# Cell 4 — Train Autoencoder on DNS aggregated features (snapshot AE)
# خروجی: agg_features با ستون ae_score
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from tensorflow.keras import layers, Model

print("🤖 Building AE dataset from agg_features...")

# --------- 1) انتخاب فیچرهای عددی (بدون leakage)
DROP_COLS = ["source.ip", "window_start", "dns_beacon_best_domain"]  # string cols
feature_cols = [c for c in agg_features.columns if c not in DROP_COLS]

# فقط عددی‌ها
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(agg_features[c])]
X = agg_features[num_cols].copy()

# NaN safety
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

print("✅ Numeric feature count:", len(num_cols))
print("✅ X shape:", X.shape)

# --------- 2) Scale
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# --------- 3) Train/Val split
X_train, X_val = train_test_split(X_scaled, test_size=0.2, random_state=42)

# --------- 4) AE model
input_dim = X_scaled.shape[1]
encoding_dim = max(8, input_dim // 2)

inp = layers.Input(shape=(input_dim,))
x = layers.Dense(encoding_dim, activation="relu")(inp)
x = layers.Dropout(0.1)(x)
x = layers.Dense(max(4, encoding_dim // 2), activation="relu")(x)
x = layers.Dense(encoding_dim, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

ae = Model(inp, out)
ae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

# --------- 5) Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = ae.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=90,
    batch_size=512,
    verbose=1,
    callbacks=callbacks
)

# --------- 6) AE score (reconstruction error)
X_pred = ae.predict(X_scaled, batch_size=1024, verbose=0)
ae_score = np.mean((X_scaled - X_pred) ** 2, axis=1)

agg_features["ae_score"] = ae_score

print("✅ AE done. ae_score stats:")
print(agg_features["ae_score"].describe(percentiles=[0.9, 0.95, 0.99, 0.995]))

display(agg_features.sort_values("ae_score", ascending=False)[["source.ip","window_start","ae_score","dns_beacon_best_domain","dns_beacon_score"]].head(10))


🤖 Building AE dataset from agg_features...
✅ Numeric feature count: 37
✅ X shape: (60578, 37)
Epoch 1/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 12.5942 - val_loss: 7.0432
Epoch 2/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.7319 - val_loss: 3.4704
Epoch 3/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 6.6508 - val_loss: 2.6695
Epoch 4/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.8939 - val_loss: 2.3704
Epoch 5/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.6383 - val_loss: 2.0808
Epoch 6/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4.5408 - val_loss: 1.7946
Epoch 7/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.7854 - val_loss: 1.5321
Epoch 8/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.9391 - val_loss: 1.3648
Epoch 9/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.8729 - val_loss: 1.2001
Epoch 10/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.2076 - val_loss: 1.1192
Epoch 11/90
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.360

,source.ip,window_start,ae_score,dns_beacon_best_domain,dns_beacon_score
16189,192.168.100.119,2025-09-20 13:24:00,567.337334,itp.local,1.000000
5154,10.10.10.35,2025-10-08 14:16:00,472.943302,google.com,0.565752
6098,10.10.10.41,2025-09-20 13:30:00,408.282047,_tcp.local,0.796715
48647,192.168.100.5,2025-09-20 13:24:00,90.174300,newrocktech.com,0.889759
2225,10.10.10.200,2025-09-20 11:21:00,86.958758,_udp.local,0.819110
1777,10.10.10.189,2025-09-22 12:29:00,81.433360,3gppnetwork.org,0.200000
17336,192.168.100.119,2025-09-30 15:19:00,73.619733,itp.local,1.000000
2504,10.10.10.200,2025-09-27 11:57:00,71.833489,apple-dns.net,0.633082
2703,10.10.10.200,2025-10-01 11:01:00,68.551042,_tcp.local,0.632679
2005,10.10.10.189,2025-10-12 12:32:00,65.139169,khosroprtgswreqetdgsrrpi-esoiiisolfinwfsksjvwu...,0.100000


In [13]:
# ==========================================================
# Cell 5 — DNS LSTM sequence model (FIXED + robust mapping)
# خروجی: agg_features با ستون lstm_score (درست و غیر صفر)
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, Model

print("🧠 Building sequences for DNS LSTM...")

# -------------------------
# 1) Sequence-friendly features
# -------------------------
SEQ_COLS = [
    "query_count",
    "uniq_queries",
    "uniq_subdomains",
    "uniq_base_domains",
    "nxdomain_rate",
    "sub_entropy_mean",
    "sum_query_len",
    "sum_query_len_roll_60m",
    "dns_beacon_score"
]

missing = [c for c in SEQ_COLS if c not in agg_features.columns]
if missing:
    raise SystemExit(f"⛔ Missing SEQ_COLS in agg_features: {missing}")

# make a clean seq_df
seq_df = agg_features[["source.ip", "window_start"] + SEQ_COLS].copy()

# normalize keys
seq_df["source.ip"] = seq_df["source.ip"].astype("string").str.strip()
seq_df["window_start"] = pd.to_datetime(seq_df["window_start"], errors="coerce")
seq_df = seq_df.dropna(subset=["window_start"]).sort_values(["source.ip", "window_start"]).reset_index(drop=True)

# NaN/inf safety
for c in SEQ_COLS:
    seq_df[c] = seq_df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# -------------------------
# 2) Scale
# -------------------------
scaler_seq = RobustScaler()
seq_df_scaled = seq_df.copy()
seq_df_scaled[SEQ_COLS] = scaler_seq.fit_transform(seq_df_scaled[SEQ_COLS].values)

# -------------------------
# 3) Build sequences (lookback=20)
# X: (N, 20, F)  y: (N, F)
# -------------------------
LOOKBACK = 20

X_list, y_list = [], []
key_ip, key_ws = [], []

for ip, g in seq_df_scaled.groupby("source.ip", sort=False):
    g = g.sort_values("window_start")
    vals = g[SEQ_COLS].values
    ws = g["window_start"].values

    if len(g) <= LOOKBACK:
        continue

    for i in range(LOOKBACK, len(g)):
        X_list.append(vals[i-LOOKBACK:i, :])
        y_list.append(vals[i, :])
        key_ip.append(ip)
        key_ws.append(ws[i])

X_seq = np.asarray(X_list, dtype=np.float32)
y_seq = np.asarray(y_list, dtype=np.float32)

print("✅ Sequences built:", X_seq.shape, "Targets:", y_seq.shape)

if X_seq.shape[0] == 0:
    raise SystemExit("⛔ No sequences were built. Increase data or reduce LOOKBACK.")

# -------------------------
# 4) Train/Val split
# -------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

# -------------------------
# 5) LSTM model
# -------------------------
F = X_seq.shape[2]

inp = layers.Input(shape=(LOOKBACK, F))
x = layers.LSTM(64, return_sequences=False)(inp)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation="relu")(x)
out = layers.Dense(F, activation="linear")(x)

lstm_model = Model(inp, out)
lstm_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=256,
    verbose=1,
    callbacks=callbacks
)

# -------------------------
# 6) Score = prediction error
# -------------------------
y_pred = lstm_model.predict(X_seq, batch_size=1024, verbose=0)
lstm_err = np.mean((y_seq - y_pred) ** 2, axis=1).astype(float)  # shape (N,)

# -------------------------
# 7) Assign back to agg_features (robust overwrite)
# - remove any old lstm_score and rebuild cleanly
# -------------------------
if "lstm_score" in agg_features.columns:
    agg_features = agg_features.drop(columns=["lstm_score"])

score_df = pd.DataFrame({
    "source.ip": pd.Series(key_ip, dtype="string").str.strip(),
    "window_start": pd.to_datetime(key_ws),
    "lstm_score": lstm_err
})

# If duplicates exist, keep worst-case (max error)
score_df = score_df.groupby(["source.ip", "window_start"], as_index=False)["lstm_score"].max()

# normalize keys in agg_features too
agg_features["source.ip"] = agg_features["source.ip"].astype("string").str.strip()
agg_features["window_start"] = pd.to_datetime(agg_features["window_start"], errors="coerce")

# left merge (rows without sequences get NaN -> 0)
agg_features = agg_features.merge(score_df, on=["source.ip", "window_start"], how="left")
agg_features["lstm_score"] = agg_features["lstm_score"].fillna(0.0)

print("✅ lstm_score stats (non-zero rows):")
nz = agg_features.loc[agg_features["lstm_score"] > 0, "lstm_score"]
print(nz.describe(percentiles=[0.9, 0.95, 0.99, 0.995]))

display(
    agg_features.sort_values("lstm_score", ascending=False)[
        ["source.ip","window_start","lstm_score","dns_beacon_best_domain","dns_beacon_score","sum_query_len","sub_entropy_mean","nxdomain_rate"]
    ].head(10)
)


🧠 Building sequences for DNS LSTM...
✅ Sequences built: (59337, 20, 9) Targets: (59337, 9)
Epoch 1/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 19s 90ms/step - loss: 4.4857 - val_loss: 3.8908
Epoch 2/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - loss: 4.2532 - val_loss: 3.8517
Epoch 3/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - loss: 3.7879 - val_loss: 3.8414
Epoch 4/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - loss: 3.8049 - val_loss: 3.8379
Epoch 5/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - loss: 3.6460 - val_loss: 3.8973
Epoch 6/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - loss: 3.6156 - val_loss: 3.8383
Epoch 7/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - loss: 3.2053 - val_loss: 3.8469
Epoch 8/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - loss: 4.0329 - val_loss: 3.8292
Epoch 9/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - loss: 3.6002 - val_loss: 3.8540
Epoch 10/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - loss: 3.7738 - val_loss: 3.8396
Epoch 11/25
186/186 ━━━━━━━━━━━━━━━━━

,source.ip,window_start,lstm_score,dns_beacon_best_domain,dns_beacon_score,sum_query_len,sub_entropy_mean,nxdomain_rate
48647,192.168.100.5,2025-09-20 13:24:00,8519.787109,newrocktech.com,0.889759,78292,3.871640,0.303571
16189,192.168.100.119,2025-09-20 13:24:00,5780.707031,itp.local,1.000000,25902,3.738418,0.893333
49791,192.168.100.5,2025-09-30 15:19:00,3572.603271,google.com,0.873092,49984,4.111458,0.300733
52173,192.168.100.5,2025-10-20 14:38:00,3512.198975,digikala.com,0.864193,50008,3.831397,0.236170
51470,192.168.100.5,2025-10-11 14:52:00,2998.346680,google.com,0.857143,45162,4.073080,0.290237
51420,192.168.100.5,2025-10-11 14:02:00,2958.161621,google.com,0.872021,44790,4.080160,0.304813
49292,192.168.100.5,2025-09-27 11:26:00,2583.885498,cymru.com,0.675847,42500,4.028193,0.298295
17336,192.168.100.119,2025-09-30 15:19:00,2304.342285,itp.local,1.000000,17334,3.556364,0.777070
51696,192.168.100.5,2025-10-12 14:38:00,2265.163574,windowsupdate.com,0.859307,40176,4.079539,0.288690
13440,192.168.10.15,2025-10-05 13:11:00,2107.318848,google.com,0.799774,26640,2.104226,0.006515


In [15]:
# ==========================================================
# Cell 6 (Improved) — DNS Fuse + Severity(top 0.5%) + Better Labeling
# Output: dns_ml_scored_severity.csv
# ==========================================================
import numpy as np
import pandas as pd
import re

print("🎯 Fusing AE + LSTM and calibrating severity (top 0.5%) + better DNS labeling ...")

REQ = [
    "source.ip","window_start",
    "ae_score","lstm_score",
    "dns_beacon_score","dns_beacon_best_domain",
    "dns_beacon_best_persist_5m","dns_beacon_best_pair_qcount","dns_beacon_best_regularity",
    # tunneling hints (from your agg features)
    "sum_query_len","sub_entropy_mean","nxdomain_rate","uniq_subdomains"
]
miss = [c for c in REQ if c not in agg_features.columns]
if miss:
    raise SystemExit(f"⛔ Missing required columns in agg_features: {miss}")

df = agg_features.copy()

# ---------- basic cleanup ----------
df["source.ip"] = df["source.ip"].astype("string").str.strip()
df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
df = df.dropna(subset=["window_start"])

for c in ["ae_score","lstm_score","dns_beacon_score","dns_beacon_best_persist_5m","dns_beacon_best_pair_qcount","dns_beacon_best_regularity",
          "sum_query_len","sub_entropy_mean","nxdomain_rate","uniq_subdomains"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

df["dns_beacon_best_domain"] = df["dns_beacon_best_domain"].fillna("").astype("string").str.strip().str.lower()

# ---------- percentiles (robust) ----------
df["ae_pct"]   = df["ae_score"].rank(pct=True, method="average").clip(0.0, 1.0)
df["lstm_pct"] = df["lstm_score"].rank(pct=True, method="average").clip(0.0, 1.0)

# union-fuse
df["fused_pct"] = np.maximum(df["ae_pct"].values, df["lstm_pct"].values)

# ---------- enforce "only top 0.5% severity>0" ----------
TARGET_TOP = 0.005
thr = df["fused_pct"].quantile(1.0 - TARGET_TOP)

df["is_alert"] = df["fused_pct"] >= thr

den = max(1e-12, (1.0 - thr))
sev_raw = (df["fused_pct"] - thr) / den

df["severity"] = 0
df.loc[df["is_alert"], "severity"] = (1 + np.floor(98 * sev_raw[df["is_alert"]])).astype(int)
df["severity"] = df["severity"].clip(0, 99).astype(int)

# ---------- better labeling helpers ----------
df["dns_beacon_pct"] = df["dns_beacon_score"].rank(pct=True, method="average").clip(0.0, 1.0)

# Noisy / local / infra domains (reduce false positives)
NOISY_DOMAIN_PATTERNS = [
    r"^_tcp\.local$", r"^_udp\.local$", r"^_.*\.local$",
    r"\.local$", r"\.localdomain$",
    r"\.arpa$", r"in-addr\.arpa$", r"ip6\.arpa$",
    r"^wpad\.", r"^isatap\.", r"^teredo\.", r"^autodiscover\.",
]
noisy_re = re.compile("|".join(NOISY_DOMAIN_PATTERNS), re.IGNORECASE)
df["is_noisy_domain"] = df["dns_beacon_best_domain"].apply(lambda s: bool(noisy_re.search(s)) if isinstance(s, str) and s else False)

# Strong BEACONING gate (avoid tiny-sample beacons)
MIN_PERSIST_5M = 3        # at least 3 minutes present in last 5 mins
MIN_PAIR_QCOUNT = 10      # at least 10 queries for best domain in that minute-window
MIN_REGULARITY = 0.35     # regularity score cutoff (your feature)

beacon_strong = (
    (df["dns_beacon_pct"] >= 0.995) &
    (df["dns_beacon_best_persist_5m"] >= MIN_PERSIST_5M) &
    (df["dns_beacon_best_pair_qcount"] >= MIN_PAIR_QCOUNT) &
    (df["dns_beacon_best_regularity"] >= MIN_REGULARITY) &
    (~df["is_noisy_domain"])
)

# TUNNEL / LOW-SLOW hints (simple but effective)
# - lots of bytes in query names OR high entropy OR lots of uniq subdomains OR high NXDOMAIN
# thresholds are intentionally mild and then filtered by top-0.5% severity anyway.
tunnel_hints = (
    (df["sum_query_len"] >= 2000) |
    (df["sub_entropy_mean"] >= 3.2) |
    (df["uniq_subdomains"] >= 40) |
    (df["nxdomain_rate"] >= 0.25)
)

# If anomaly is high but domain is noisy/local, don't call it tunneling; keep as OTHER_ANOMALY
tunnel_candidate = (
    (~beacon_strong) &
    tunnel_hints &
    (~df["is_noisy_domain"])
)

# ---------- final label ----------
df["alert_type"] = "NONE"
mask = df["is_alert"]

df.loc[mask & beacon_strong, "alert_type"] = "BEACONING"
df.loc[mask & (~beacon_strong) & tunnel_candidate, "alert_type"] = "DNS_TUNNEL_LOW_SLOW"
df.loc[mask & (df["alert_type"] == "NONE"), "alert_type"] = "OTHER_ANOMALY"

# ---------- prints ----------
total = len(df)
alerts = int(df["is_alert"].sum())
print("✅ Total rows:", total)
print(f"✅ severity>0 rows: {alerts} ({alerts/total*100:.3f}%) target≈0.5%")
print("✅ Split (severity>0):")
print(df.loc[df["is_alert"], "alert_type"].value_counts())

# show top
cols_show = [
    "source.ip","window_start","severity","alert_type",
    "ae_score","lstm_score","ae_pct","lstm_pct","fused_pct",
    "dns_beacon_best_domain","dns_beacon_score","dns_beacon_pct",
    "dns_beacon_best_persist_5m","dns_beacon_best_pair_qcount","dns_beacon_best_regularity",
    "sum_query_len","sub_entropy_mean","nxdomain_rate","uniq_subdomains",
    "is_noisy_domain"
]
display(df.sort_values(["severity","fused_pct"], ascending=[False, False])[cols_show].head(30))

OUT = "dns_ml_scored_severity.csv"
df.to_csv(OUT, index=False)
print("📦 Saved:", OUT)


🎯 Fusing AE + LSTM and calibrating severity (top 0.5%) + better DNS labeling ...
✅ Total rows: 60578
✅ severity>0 rows: 303 (0.500%) target≈0.5%
✅ Split (severity>0):
alert_type
OTHER_ANOMALY          152
DNS_TUNNEL_LOW_SLOW    151
Name: count, dtype: int64


,source.ip,window_start,severity,alert_type,ae_score,lstm_score,ae_pct,lstm_pct,fused_pct,dns_beacon_best_domain,dns_beacon_score,dns_beacon_pct,dns_beacon_best_persist_5m,dns_beacon_best_pair_qcount,dns_beacon_best_regularity,sum_query_len,sub_entropy_mean,nxdomain_rate,uniq_subdomains,is_noisy_domain
16189,192.168.100.119,2025-09-20 13:24:00,99,OTHER_ANOMALY,567.337334,5780.707031,1.000000,0.999983,1.000000,itp.local,1.000000,0.780440,5.0,48,1.000000,25902,3.738418,0.893333,202,True
48647,192.168.100.5,2025-09-20 13:24:00,99,DNS_TUNNEL_LOW_SLOW,90.174300,8519.787109,0.999950,1.000000,1.000000,newrocktech.com,0.889759,0.510829,5.0,12,0.779519,78292,3.871640,0.303571,211,False
5154,10.10.10.35,2025-10-08 14:16:00,98,OTHER_ANOMALY,472.943302,0.000000,0.999983,0.010251,0.999983,google.com,0.565752,0.011753,4.0,2,0.414380,308,2.263839,0.000000,6,False
6098,10.10.10.41,2025-09-20 13:30:00,97,OTHER_ANOMALY,408.282047,0.000000,0.999967,0.010251,0.999967,_tcp.local,0.796715,0.297369,5.0,18,0.593430,65990,2.853362,0.195918,80,True
49791,192.168.100.5,2025-09-30 15:19:00,97,DNS_TUNNEL_LOW_SLOW,25.961593,3572.603271,0.999389,0.999967,0.999967,google.com,0.873092,0.467109,5.0,6,0.746185,49984,4.111458,0.300733,130,False
52173,192.168.100.5,2025-10-20 14:38:00,97,DNS_TUNNEL_LOW_SLOW,21.185753,3512.198975,0.999026,0.999950,0.999950,digikala.com,0.864193,0.440606,5.0,12,0.728386,50008,3.831397,0.236170,142,False
2225,10.10.10.200,2025-09-20 11:21:00,96,OTHER_ANOMALY,86.958758,0.307230,0.999934,0.657466,0.999934,_udp.local,0.819110,0.352471,5.0,6,0.638219,906,0.596951,0.000000,4,True
51470,192.168.100.5,2025-10-11 14:52:00,96,DNS_TUNNEL_LOW_SLOW,20.581871,2998.346680,0.998960,0.999934,0.999934,google.com,0.857143,0.434539,5.0,12,0.714286,45162,4.073080,0.290237,123,False
1777,10.10.10.189,2025-09-22 12:29:00,96,DNS_TUNNEL_LOW_SLOW,81.433360,0.860834,0.999917,0.793820,0.999917,3gppnetwork.org,0.200000,0.006009,2.0,2,0.000000,84,3.671406,0.000000,1,False
51420,192.168.100.5,2025-10-11 14:02:00,96,DNS_TUNNEL_LOW_SLOW,20.349113,2958.161621,0.998877,0.999917,0.999917,google.com,0.872021,0.456370,5.0,14,0.744042,44790,4.080160,0.304813,125,False


📦 Saved: dns_ml_scored_severity.csv


In [19]:
# --- Cell 7 (DNS): Enrich alerts with raw context + Download ---

from google.colab import files
import pandas as pd
import numpy as np

print("🧩 Enriching DNS alerts with raw context...")

# --------- Guards ----------
required_alert_cols = {"source.ip", "window_start"}
required_raw_cols = {"source.ip", "window_start", "ts", "qtype", "rcode", "query", "base_domain"}

missing_a = required_alert_cols - set(alerts.columns)
missing_r = required_raw_cols - set(df_clean.columns)

if missing_a:
    raise KeyError(f"alerts is missing columns: {missing_a}")
if missing_r:
    raise KeyError(f"df_clean is missing columns: {missing_r}")

# keep only alerts (severity>0)
alerts_in = alerts.copy()
if "severity" in alerts_in.columns:
    alerts_in = alerts_in[alerts_in["severity"] > 0].copy()

print(f"✅ Alerts to enrich: {len(alerts_in)}")

# --------- TTL column detection (robust) ----------
ttl_candidates = ["ttl_mean", "ttl_avg", "ttl_min", "ttl_max", "ttl"]
ttl_col = next((c for c in ttl_candidates if c in df_clean.columns), None)
if ttl_col is None:
    # not fatal; we can still enrich without TTL
    print("⚠️ No TTL column found in df_clean; mean_ttl will be NaN.")
else:
    print(f"✅ Using TTL column for context: {ttl_col}")

# --------- Helper: safe mode ----------
def safe_mode(series):
    series = series.dropna()
    if len(series) == 0:
        return ""
    vc = series.astype(str).value_counts()
    return vc.index[0] if len(vc) else ""

# --------- Pull raw rows for alerts only ----------
raw_alert = df_clean.merge(
    alerts_in[["source.ip", "window_start"]],
    on=["source.ip", "window_start"],
    how="inner"
).copy()

print(f"✅ Raw rows for alerts: {len(raw_alert)}")

# --------- Build context table ----------
agg_dict = {
    "raw_rows": ("ts", "count"),

    "top_domain": ("base_domain", safe_mode),
    "top_query": ("query", safe_mode),
    "top_qtype": ("qtype", safe_mode),
    "top_rcode": ("rcode", safe_mode),

    "uniq_queries_raw": ("query", pd.Series.nunique),
    "uniq_subdomains_raw": ("subdomain", pd.Series.nunique) if "subdomain" in raw_alert.columns else ("query", lambda s: np.nan),
    "uniq_base_domains_raw": ("base_domain", pd.Series.nunique),

    "txt_count": ("is_txt", "sum") if "is_txt" in raw_alert.columns else ("qtype", lambda s: int((s.astype(str) == "TXT").sum())),
    "any_count": ("is_any", "sum") if "is_any" in raw_alert.columns else ("qtype", lambda s: int((s.astype(str) == "ANY").sum())),
    "null_count": ("is_null", "sum") if "is_null" in raw_alert.columns else ("qtype", lambda s: int((s.astype(str) == "NULL").sum())),

    "sum_query_len_raw": ("query_len", "sum") if "query_len" in raw_alert.columns else ("query", lambda s: int(s.astype(str).str.len().sum())),
    "mean_entropy": ("sub_entropy", "mean") if "sub_entropy" in raw_alert.columns else ("subdomain_entropy", "mean") if "subdomain_entropy" in raw_alert.columns else ("query", lambda s: np.nan),
    "mean_rtt": ("rtt", "mean") if "rtt" in raw_alert.columns else ("zeek.dns.rtt", "mean") if "zeek.dns.rtt" in raw_alert.columns else ("query", lambda s: np.nan),

    "nxdomain_rate_raw": ("is_nxdomain", "mean") if "is_nxdomain" in raw_alert.columns else ("rcode", lambda s: float((s.astype(str) == "NXDOMAIN").mean())),
    "rejected_rate_raw": ("is_rejected", "mean") if "is_rejected" in raw_alert.columns else ("zeek.dns.rejected", "mean") if "zeek.dns.rejected" in raw_alert.columns else ("query", lambda s: np.nan),
    "tc_rate_raw": ("is_tc", "mean") if "is_tc" in raw_alert.columns else ("zeek.dns.TC", "mean") if "zeek.dns.TC" in raw_alert.columns else ("query", lambda s: np.nan),
}

if ttl_col is not None:
    agg_dict["mean_ttl"] = (ttl_col, "mean")
else:
    agg_dict["mean_ttl"] = ("query", lambda s: np.nan)

ctx = raw_alert.groupby(["source.ip", "window_start"], as_index=False).agg(**agg_dict)

# --------- Build a compact text context ----------
def build_context_row(r):
    parts = [
        f"top_domain={r.get('top_domain','')}",
        f"top_query={r.get('top_query','')}",
        f"top_qtype={r.get('top_qtype','')}",
        f"top_rcode={r.get('top_rcode','')}",
        f"raw_rows={int(r.get('raw_rows',0))}",
        f"uniq_domains={int(r.get('uniq_base_domains_raw',0))}",
        f"uniq_queries={int(r.get('uniq_queries_raw',0))}",
        f"nxdomain_rate={float(r.get('nxdomain_rate_raw',0)):.3f}",
        f"tc_rate={float(r.get('tc_rate_raw',0)):.3f}" if pd.notna(r.get("tc_rate_raw", np.nan)) else "tc_rate=NA",
        f"rej_rate={float(r.get('rejected_rate_raw',0)):.3f}" if pd.notna(r.get("rejected_rate_raw", np.nan)) else "rej_rate=NA",
        f"mean_rtt={float(r.get('mean_rtt',0)):.4f}" if pd.notna(r.get("mean_rtt", np.nan)) else "mean_rtt=NA",
        f"mean_ttl={float(r.get('mean_ttl',0)):.2f}" if pd.notna(r.get("mean_ttl", np.nan)) else "mean_ttl=NA",
        f"sum_query_len={int(r.get('sum_query_len_raw',0))}",
    ]
    return " | ".join(parts)

ctx["alert_context"] = ctx.apply(build_context_row, axis=1)

# --------- Merge back to alerts ----------
out = alerts_in.merge(ctx, on=["source.ip", "window_start"], how="left")

# --------- Save + Download ----------
OUTFILE = "dns_ml_scored_severity_enriched.csv"
out.to_csv(OUTFILE, index=False, encoding="utf-8")

print(f"📦 Saved: {OUTFILE}")
print("⬇️ Download starting...")
files.download(OUTFILE)


🧩 Enriching DNS alerts with raw context...
✅ Alerts to enrich: 303
✅ Using TTL column for context: ttl_avg
✅ Raw rows for alerts: 99443
📦 Saved: dns_ml_scored_severity_enriched.csv
⬇️ Download starting...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>